### 🛠️ Cell 1: Environment Setup & Hardware Acceleration
**Function:** Imports necessary libraries, fixes potential system duplicate library crashes, and initializes the NVIDIA GPU (CUDA) if available to speed up processing.

In [ ]:
!pip install ultralytics

# =====================================================================
# 📁 CELL 1 — GOOGLE DRIVE MOUNTING & PATH CONFIGURATION
# =====================================================================
# DESCRIPTION: This cell connects your Google Colab session to your
# personal Google Drive storage to fetch the video file directly.
# =====================================================================

print("[INFO] Requesting Google Drive access permission...")

from google.colab import drive

# ⚠️ CRITICAL STEP 1: Execute this line to mount your Google Drive.
# A pop-up will appear asking for permissions; click "Connect to Google Drive" and select your account.
drive.mount('/content/drive')

print("[SUCCESS] Google Drive mounted successfully!")

In [ ]:

import os
import time
import cv2
import torch
import numpy as np
from ultralytics import YOLO

# 1. Initialize environment and resolve system library duplicate issues
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# 2. Enable GPU hardware acceleration if available
if torch.cuda.is_available():
    torch.cuda.set_device(0)
    print(f"[INFO] Hardware acceleration enabled on: {torch.cuda.get_device_name(0)}")
else:
    print("[WARNING] CUDA GPU not detected. Using CPU.")

### 🧠 Cell 2: Deep Learning Model Initialization
**Function:** Loads the YOLOv8 weights into the active memory. Keeping this in a separate cell prevents reloading the heavy model every time you want to test the video.

In [ ]:
# =====================================================================
# 🧠 CELL 2 — MODEL ARCHITECTURE LOADING
# =====================================================================
# 3. Load YOLO model weights for object detection and tracking
model = YOLO("yolo26l.pt", task="detect")
print("[SUCCESS] YOLO Model loaded successfully.")

### ⚙️ Cell 3: Configuration & File Paths
**Function:** Initializes global tracking variables, counters, and sets up the input/output file paths. **(Update your video path here)**.

In [ ]:
# =====================================================================
# ⚙️ CELL 3 — CONFIGURATION & METRICS INITIALIZATION
# =====================================================================
# Global counters and state dictionaries
total_in = 0
total_out = 0
track_history = {}
crossed_direction = {}
entry_timestamps = {}
dwell_durations = {}

# =====================================================================
# 🔴 🚨 URGENT: CHANGE THE VIDEO PATH HERE 🚨 🔴
# =====================================================================
# Replace "*************" with the exact path to your local video.
video_file_path = "********************"  # <--- INSERT YOUR VIDEO PATH HERE

output_file_path = "output_analytics.mp4"
window_name = "OFFLINE VIDEO ANALYSIS"

print(f"[INFO] Target video file configured: {video_file_path}")

# =====================================================================
# 🔍 CELL 3.5 — COLAB INTERACTIVE COORDINATE FINDER (PLOTLY VERSION)
# =====================================================================

In [ ]:
import cv2
import plotly.express as px
import numpy as np

print("[INFO] Loading interactive preview for Google Colab...")

# Open the video file temporarily
preview_cap = cv2.VideoCapture(video_file_path)

if not preview_cap.isOpened():
    print(f"[ERROR] Cannot open '{video_file_path}'. Please check your path in Cell 3!")
else:
    # Jump to second 5 (5000ms) to see the actual scene with walking space
    preview_cap.set(cv2.CAP_PROP_POS_MSEC, 5000)
    ret, preview_frame = preview_cap.read()
    preview_cap.release()  # Close the file immediately

    if ret and preview_frame is not None:
        # Convert BGR (OpenCV) to RGB (Plotly/Normal format)
        preview_rgb = cv2.cvtColor(preview_frame, cv2.COLOR_BGR2RGB)

        # Create an interactive Plotly image
        fig = px.imshow(preview_rgb)

        # Update layout to make it big and clear
        fig.update_layout(
            width=900,
            height=600,
            title="Colab Interactive Preview: Move your mouse to see X and Y coordinates!"
        )

        # Display the interactive plot inside the notebook
        fig.show()

        print("[SUCCESS] Look at the image above! Move your mouse over it to see exact (x, y) pixels.")
    else:
        print("[ERROR] Failed to extract the frame at the specified timestamp.")

### 🎬 Cell 4: Main Video Processing & Analytics Pipeline
**Function:** Opens the video file, runs frame-by-frame YOLO tracking, calculates crossing logic and dwell times, renders the UI overlays, and exports the final MP4 video.

In [ ]:
# =====================================================================
# 🚀 CELL 4 — CORE VIDEOS PROCESSING & OCCUPANCY ANALYTICS
# =====================================================================
# DESCRIPTION: This cell loads the YOLO model, tracks people, processes
# crossing logic (Horizontal/Vertical/Polygon), and saves the rendered video.
# =====================================================================

import cv2
import os
import time
import numpy as np
from ultralytics import YOLO

# ⚡ ⚡ [LOCAL PC NOTE] TERMINAL COMMAND FOR LOCAL INSTALLATION (RTX 4060 / CUDA 12.4):
# If you are running this code on your local PC with your RTX 4060 GPU, run this command in your terminal first:
# pip install ultralytics opencv-python numpy torch torchvision --index-url https://download.pytorch.org/whl/cu124

print("[INFO] Starting core video processing with updated Top Bar Dashboard...")

# 1. Open the video file using OpenCV
cap = cv2.VideoCapture(video_file_path)

if not cap.isOpened():
    print(f"[ERROR] Cannot open video file: '{video_file_path}'. Please check Cell 3.")
else:
    # 2. Get video properties for dimensions and frame rate
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"[INFO] Video Resolution: {w}x{h} | FPS: {fps:.2f} | Total Frames: {total_frames}")

    # 3. Initialize Output Video Writer (Saves directly to Colab files)
    output_file_path = "output_analytics.mp4"
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_file_path, fourcc, fps, (w, h))

    # 4. Load the trained YOLO model
    model_path = "yolo26l.pt" if os.path.exists("yolo26l.pt") else "yolov8n.pt"
    model = YOLO(model_path)

    # 5. Tracking, Timing and Counting Data Structures
    user_states = {}         # Stores whether a track_id is "IN" or "OUT"
    entry_times = {}         # Stores the timestamp when a track_id crossed "IN"
    completed_durations = [] # List of durations for people who left

    # Global counters for cumulative statistics
    total_in_counter = 0
    total_out_counter = 0

    # =====================================================================
    # 📐 LINE CONFIGURATION TUNING BLOCK (USER CONFIGURABLE)
    # =====================================================================
    # ─── ACTIVE DEFAULT: HORIZONTAL LINE ─────────────────────────────────
    x1, y1 = 0, 135      # Start point (Left edge, height = 135px)
    x2, y2 = w, 135      # End point (Right edge, height = 135px)
    detection_mode = "horizontal_line" 

    # ─── ALTERNATIVE OPTIONS (UNCOMMENT TO ACTIVATE & COMMENT OUT ABOVE) ──
    #
    # --- Option B: Vertical Line Detection ---
    # To use: Comment out the 3 lines of "HORIZONTAL LINE" above, and uncomment the 3 lines below:
    # line_x_pixel = int(w * 0.50)
    # detection_mode = "vertical_line"
    #
    # --- Option C: Polygon-based Detection ---
    # To use: Comment out the 3 lines of "HORIZONTAL LINE" above, and uncomment the 3 lines below:
    # polygon_points = np.array([[200, 200], [800, 200], [800, 500], [200, 500]], np.int32)
    # detection_mode = "polygon"
    # =====================================================================

    frame_count = 0
    start_processing_time = time.time()

    # 6. Main Video Frame Processing Loop
    while cap.isOpened():
        t_frame_start = time.time() # To calculate live FPS per frame
        ret, frame = cap.read()
        if not ret:
            break  # End of video reached

        frame_count += 1
        current_video_time = frame_count / fps  # Current time in seconds within the video

        # =====================================================================
        # 🎨 DRAW DETECTION REGION BASED ON ACTIVE MODE
        # =====================================================================
        if detection_mode == "horizontal_line":
            cv2.line(frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
            
        elif detection_mode == "vertical_line":
            cv2.line(frame, (line_x_pixel, 0), (line_x_pixel, h), (0, 0, 255), 3)
            
        elif detection_mode == "polygon":
            cv2.polylines(frame, [polygon_points], isClosed=True, color=(0, 255, 255), thickness=3)
        # =====================================================================

        # Run YOLO Tracking pipeline on the current frame
        results = model.track(frame, persist=True, verbose=False, classes=[0]) # class 0 is Person

        # Check if any people are detected and tracked
        if results[0].boxes is not None and results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            track_ids = results[0].boxes.id.cpu().numpy().astype(int)

            for box, track_id in zip(boxes, track_ids):
                # Calculate the center X and bottom Y (feet level) of the person
                center_x = int((box[0] + box[2]) / 2)
                center_y = int(box[3])

                # =====================================================================
                # 🧮 GENERAL CROSSING LOGIC SWITCHBOARD
                # =====================================================================
                if detection_mode == "horizontal_line":
                    current_state = "IN" if center_y > y1 else "OUT"
                    
                elif detection_mode == "vertical_line":
                    current_state = "IN" if center_x > line_x_pixel else "OUT"
                    
                elif detection_mode == "polygon":
                    is_inside = cv2.pointPolygonTest(polygon_points, (center_x, int(box[3])), False) >= 0
                    current_state = "IN" if is_inside else "OUT"
                # =====================================================================

                # State Machine: Check for state changes (Crossing the Line)
                if track_id in user_states:
                    previous_state = user_states[track_id]

                    # Scenario 1: Person just crossed from OUT to IN (Entered)
                    if previous_state == "OUT" and current_state == "IN":
                        entry_times[track_id] = current_video_time
                        total_in_counter += 1  # Increment cumulative entrance counter

                    # Scenario 2: Person just crossed from IN to OUT (Exited)
                    elif previous_state == "IN" and current_state == "OUT":
                        total_out_counter += 1 # Increment cumulative exit counter
                        if track_id in entry_times:
                            duration = current_video_time - entry_times[track_id]
                            completed_durations.append(duration)
                            del entry_times[track_id] # Clean up entry record
                else:
                    # First time seeing this person, initialize their state
                    if current_state == "IN":
                        entry_times[track_id] = current_video_time
                        total_in_counter += 1

                # Save the current state for the next frame evaluation
                user_states[track_id] = current_state

                # Visual Feedback: Bounding box color (Green if IN, Blue if OUT)
                box_color = (0, 255, 0) if current_state == "IN" else (255, 0, 0)

                # Draw bounding box and tracking ID label
                cv2.rectangle(frame, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), box_color, 2)
                cv2.putText(frame, f"ID: {track_id} ({current_state})", (int(box[0]), int(box[1]) - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, box_color, 2)

        # 7. Calculate real-time metrics
        active_inside_count = sum(1 for state in user_states.values() if state == "IN")
        avg_duration = sum(completed_durations) / len(completed_durations) if completed_durations else 0.0

        # Calculate process FPS for the frame
        t_frame_end = time.time()
        live_fps = 1.0 / (t_frame_end - t_frame_start) if (t_frame_end - t_frame_start) > 0 else fps

        # 📊 FULL-WIDTH TOP BAR DASHBOARD DESIGN
        # Create a solid black banner at the very top of the screen (Height: 45 pixels)
        cv2.rectangle(frame, (0, 0), (w, 45), (0, 0, 0), -1)
        # Add a subtle gray border separator line at the bottom of the banner
        cv2.line(frame, (0, 45), (w, 45), (100, 100, 100), 1)

        # Structure the statistical text in a single elegant line across the banner width
        dashboard_text = (
            f"Active: {active_inside_count}  |  "
            f"Total IN: {total_in_counter}  |  "
            f"Total OUT: {total_out_counter}  |  "
            f"Avg Dwell: {avg_duration:.1f}s  |  "
            f"FPS: {live_fps:.1f}"
        )

        # Write the metric ribbon on the top bar
        cv2.putText(frame, dashboard_text, (20, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 2, cv2.LINE_AA)

        # 8. Write the finalized processed frame directly to the output video file
        out.write(frame)

        # ⚡ ⚡ [LOCAL PC NOTE] LIVE PREVIEW WINDOW:
        # The 3 lines below create a live popup window to see the video processing in real-time.
        # This DOES NOT WORK ON GOOGLE COLAB (it will crash the notebook), but works perfectly on Local PC!
        # To use it locally, just remove the '#' from the 3 lines below:
        # cv2.imshow("Local PC Live Analytics Preview", frame)
        # if cv2.waitKey(1) & 0xFF == ord('q'):
        #     break

        # Print progress update every 50 frames to monitor execution in console
        if frame_count % 50 == 0:
            progress_pct = (frame_count / total_frames) * 100
            print(f"[PROCESSING] Completed {frame_count}/{total_frames} frames ({progress_pct:.1f}%)")

    # 9. Clean up and close all assets
    cap.release()
    out.release()
    
    # ⚡ ⚡ [LOCAL PC NOTE] CLOSING WINDOWS:
    # This line closes the popup live window on your PC. Comment it out if you are on Google Colab.
    # cv2.destroyAllWindows()

    total_processing_time = time.time() - start_processing_time
    print("-" * 70)
    print(f"[SUCCESS] Video analysis compiled fully in {total_processing_time:.2f} seconds.")
    print(f"[SUCCESS] Rendered file saved safely as: '{output_file_path}'")
    print("-" * 70)

# =====================================================================
# 🎬 CELL 5 — CONVERT VIDEO TO STANDARD H.264 & INITIATE DOWNLOAD
# =====================================================================

In [ ]:
# =====================================================================
# 🎬 CELL 5 — CONVERT VIDEO TO STANDARD H.264 & INITIATE DOWNLOAD
# =====================================================================
# ⚡ ⚡ [LOCAL PC NOTE] COLAB ONLY CELL:
# This entire cell is designed only for Google Colab to safely format and trigger an automatic 
# browser download pop-up. If you are running this on your Local PC, you can completely COMMENT OUT 
# or delete this cell because your video is already saved directly in your project folder!
# =====================================================================

print("[1/2] Converting video codec to standard H.264...")
# !ffmpeg -y -i output_analytics.mp4 -vcodec libx264 -f mp4 final_fixed_video.mp4

print("[2/2] Launching direct browser download window...")
# from google.colab import files
# files.download('final_fixed_video.mp4')